# Programme 2 - Cases a cocher & grilles de bulles
**Bas niveau** : on segmente chaque case carree par `regionprops` (densite de remplissage > 0.82, rapport hauteur/largeur ~ 1, taille proche de la mediane). L'etat d'une case est lu par la **densite d'encre interieure** sur l'image en niveaux de gris (vide ~ 0.0 ; cochee 'X' ~ 0.4-0.6 ; **noircie = correction = NON cochee** ~ 1.0, conformement aux instructions du formulaire). La geometrie de la grille est obtenue par **clustering par trous** (`_gap_cluster`) qui suit la distorsion du scan, sans grille rigide.

In [ ]:
# --- Choix du dossier, du PDF et de la page ------------------------------
# dossier    : "FORM1" / "FORM2" / "FORM3"
# page_index : 0 = page d'identification (PAGE-01) ; 4 = 1re page de questions
import os, glob, numpy as np
import reader_pdf as R

dossier    = "FORM1"
page_index = 0

pdfs = sorted(glob.glob(os.path.join(dossier, "*.pdf")))
if not pdfs:
    raise FileNotFoundError(
        f"Aucun PDF dans '{os.path.abspath(dossier)}'. "
        f"Lance Jupyter depuis la racine du projet, ou mets un chemin complet.")
pdf_path = pdfs[0]
print("PDF utilise :", pdf_path, "| total trouves :", len(pdfs))

pages = R.pdf_to_pages(pdf_path, dpi=200)   # 1 image par page (pdf2image -> fitz en secours)
im    = pages[page_index]
print("nb pages :", len(pages), "| page", page_index, "shape", im.shape)


In [ ]:
import matplotlib.pyplot as plt
g = R.norm_gray(im)   # page recalee (gris)
H, W = g.shape

# --- STUDENT ID : grille 5 colonnes x 10 lignes ---------------------------
sid = R.read_digit_grid(g, 0.81, 0.99, 0.18, 0.45, 5)
print("STUDENT ID lu :", sid)

# --- GROUP : 2 colonnes de chiffres + 1 colonne de lettres (A-J) ----------
grp = R.read_group_grid(g, 0.49, 0.70, 0.185, 0.45)
print("GROUP lu      :", grp)

## Visualisation : cases detectees et etat (vert=vide, rouge=cochee, orange=correction)

In [ ]:
from skimage.draw import rectangle_perimeter
x0,x1,y0,y1 = 0.81,0.99,0.18,0.45
reg = g[int(y0*H):int(y1*H), int(x0*W):int(x1*W)]
boxes = R.detect_boxes(reg, size=3)
vis = np.stack([reg]*3, axis=-1)
for cx,cy,(r0,c0,r1,c1) in boxes:
    v = R.interior_ink(reg,(r0,c0,r1,c1))
    col = [1,0,0] if 0.12<v<0.85 else ([1,0.6,0] if v>=0.85 else [0,0.7,0])
    rr,cc = rectangle_perimeter((r0,c0),(r1-1,c1-1),shape=reg.shape)
    vis[rr,cc]=col
plt.figure(figsize=(4,7)); plt.imshow(vis); plt.title(f"Grille STUDENT ID -> {sid}"); plt.axis('off'); plt.show()
print("cases detectees :", len(boxes))

## Conditions d'examen (remplissage gris plein = case selectionnee)

In [ ]:
# Bande des 5 paires YES/NO ; la case selectionnee est remplie -> plus d'encre.
y0c, y1c = 0.60, 0.645
regc = g[int(y0c*H):int(y1c*H), :]
bx = sorted(R.detect_boxes(regc, size=3, mn=30, mind=10, dens=0.6), key=lambda b:b[0])
inks = [R.interior_ink(regc, b[2], sh=0.2, t=0.65) for b in bx[:10]]
cond = [1 if inks[i] >= inks[i+1] else 0 for i in range(0, 10, 2)]
print("CONDITIONS [1=YES,0=NO] :", cond)